# Data processing

The goal of the model is to determine from a single board position -- a snapshot a game in progress -- what the likely outcome of the game will be. The first step of this is to generate the snapshots, and then do preprocessing.

Here is the rough process we will follow:
1. For each game, pick one random ply (excluding the first ~6 and the final position), because it is either too early to say or too obvious
2. We extract a 12x8x8 board tensor (flattened to 768 floats), normalized elo for white, black, the total amount of material remaining from
   black and white, and finally, ply -- which is the move number of the current position.
3. The label will be one-hot encoded, a categorical value in `0, 1, 2` for black win, draw, or white win.
4. We will write the resulting data to parquet files, chunked at 1 million rows.

In [3]:
import duckdb

In [2]:
# Sample code from before

duckdb.sql("""
SELECT
  lc.* EXCLUDE (movedata, clocks_white, clocks_black, tournament),                -- consider listing columns instead of *
  t.ply,
  fen_at_position(lc.movedata, t.ply) AS fen
FROM subset AS lc
CROSS JOIN LATERAL (
  SELECT 1 + CAST(floor(random() * lc.ply_count) AS INTEGER) AS ply
) AS t
""").df()

NameError: name 'duckdb' is not defined